# Task: GNN training on synthetic SPV simulations: adjacency, cell state and property matrices

We will be developing a graph neural network (GNN)-based model capable of inferring mechanistic rules and uncovering the principles driving DPAC aggregation. To facilitate this, the GNN will initially be trained using synthetic Self-Propelled Voronoi (SPV) simulations, serving as placeholder data while the deep learning infrastructure is optimized. The GNN will be validated by its ability to, first, recover the physical mechanisms embedded in the SPV model, then subsequently applied to DPAC data to explore the impacts of initial thickness and cell density.

### GNN training

In [ ]:

import networkx as nx
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

from pysr import PySRRegressor
import numpy as np
import torch.nn as nn
from torch.optim import Adam
from torch_geometric.utils import from_networkx, add_self_loops
from tqdm import tqdm
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.nn import MessagePassing
from sklearn.model_selection import KFold


In [2]:
import os
import networkx as nx
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F  # Added for loss functions
from torch.optim import Adam, AdamW, lr_scheduler  # Added AdamW
from torch_geometric.utils import from_networkx
from torch_geometric.data import DataLoader  # Added for data loading
from torch_geometric.nn import MessagePassing
from sklearn.model_selection import KFold

In [83]:
import os
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW, lr_scheduler
from torch_geometric.utils import from_networkx
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv
from torch.utils.data import random_split
from sklearn.model_selection import KFold

# ----------------------------
# 1) Parameter Parsing
# ----------------------------
def parse_parameters(param_path):
    """
    Parse parameters from the given file.
    """
    params = {}
    print(f"Parsing parameters from {param_path}")
    with open(param_path, "r") as f:
        exec(f.read(), {}, params)  # Execute the file content in a controlled namespace

    if not isinstance(params["W"], np.ndarray):
        params["W"] = np.array(params["W"])  # Convert W to NumPy array

    print(f"Successfully parsed parameters: {params}")
    return params

# ----------------------------
# 2) Loading Matrices
# ----------------------------
def load_matrices(directory, timepoint):
    """
    Load matrices with caching for faster subsequent access
    """
    cache_key = (directory, timepoint)
    if not hasattr(load_matrices, 'cache'):
        load_matrices.cache = {}
    if cache_key not in load_matrices.cache:
        print(f"Loading matrices for timepoint {timepoint} from {directory}")
        graph_mat = np.load(os.path.join(directory, f"{timepoint}_graph_mat.npy"))
        properties_mat = np.load(os.path.join(directory, f"{timepoint}_properties_mat.npy"))
        state_mat = np.load(os.path.join(directory, f"{timepoint}_state_mat.npy"))
        load_matrices.cache[cache_key] = (graph_mat, properties_mat, state_mat)
    return load_matrices.cache[cache_key]

# ----------------------------
# 3) GCA Graph Initialization
# ----------------------------
def initialize_gca(graph_mat, properties_mat, state_mat, params):
    """
    Create graph with proper state handling and parameter validation.
    Stores relevant attributes in each node (area, perimeter, etc.)
    and edge (adhesion, repulsion_radius, ...).
    """
    print("Initializing GCA graph...")
    g = nx.from_numpy_array(graph_mat, create_using=nx.Graph)

    # Validate physical constraints
    if np.any(properties_mat[:, 0] < 0) or np.any(properties_mat[:, 1] < 0):
        raise ValueError("Area and perimeter must be non-negative")

    # Add node properties with proper state handling
    print("Adding node properties...")
    for i, (area, perimeter) in enumerate(properties_mat):
        if area < 0 or perimeter < 0:
            raise ValueError(f"Invalid properties at node {i}: area={area}, perimeter={perimeter}")

        # cell_type is the argmax of the state distribution
        cell_type = np.argmax(state_mat[i])
        g.nodes[i]["state"] = state_mat[i]  # Full state distribution
        g.nodes[i]["area"]  = max(area, 0)
        g.nodes[i]["perimeter"] = max(perimeter, 0)

        # Additional parameters from 'params'
        g.nodes[i]["motility"]    = params["v0"][cell_type]
        g.nodes[i]["persistence"] = params["Dr"]
        g.nodes[i]["kappa_A"]     = params["kappa_A"]
        g.nodes[i]["kappa_P"]     = params["kappa_P"]
        g.nodes[i]["A0"]          = params["A0"][cell_type]
        g.nodes[i]["P0"]          = params["P0"][cell_type]

    # Add edge properties with validation
    print("Adding edge properties...")
    for u, v in g.edges():
        type_u = np.argmax(g.nodes[u]["state"])
        type_v = np.argmax(g.nodes[v]["state"])
        adhesion = params["W"][type_u][type_v]  # adhesion from W matrix
        g.edges[u, v]["adhesion"] = adhesion
        g.edges[u, v]["repulsion_radius"] = params["a"]
        g.edges[u, v]["repulsion_coefficient"] = params["k"]

    g.graph["adj_matrix"] = graph_mat
    print("GCA graph initialization complete.")
    return g

# ----------------------------
# 4) GNN Definition
# ----------------------------
class GraphPredictor(nn.Module):
    """
    A Graph Neural Network that:
      1) Takes node-level features of dimension = node_dim
      2) Takes edge-level features of dimension = edge_dim
      3) Produces four outputs:
         - state_pred  (log-probs over num_cell_types)
         - area_pred   (scalar)
         - perim_pred  (scalar)
         - adj_pred    (scalar per edge, for adjacency)
    """
    def __init__(self, node_dim, edge_dim, hidden_dim, num_cell_types):
        super(GraphPredictor, self).__init__()

        # --- GNN Layers ---
        self.conv1 = GCNConv(node_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        # --- Node-level Heads ---
        self.state_head = nn.Linear(hidden_dim, num_cell_types)  # for cell-type distribution
        self.area_head  = nn.Linear(hidden_dim, 1)               # scalar
        self.perim_head = nn.Linear(hidden_dim, 1)               # scalar

        # --- Edge-level Head ---
        # Predict adjacency from [h_u, h_v, edge_attr].
        # Output is a single scalar (use BCEWithLogitsLoss).
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, edge_index, edge_attr):
        """
        Args:
            x          : [N, node_dim]
            edge_index : [2, E]
            edge_attr  : [E, edge_dim]

        Returns:
            state_pred : [N, num_cell_types] (log-probabilities)
            area_pred  : [N, 1]
            perim_pred : [N, 1]
            adj_pred   : [E, 1]
        """
        # --- 1) Node Embedding ---
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))

        # --- 2) Node-level predictions ---
        state_pred  = F.log_softmax(self.state_head(h), dim=-1)  # for KLDivLoss
        area_pred   = self.area_head(h)
        perim_pred  = self.perim_head(h)

        # --- 3) Edge-level predictions ---
        row, col = edge_index
        edge_inputs = torch.cat([h[row], h[col], edge_attr], dim=-1)
        adj_pred    = self.edge_mlp(edge_inputs)

        return state_pred, area_pred, perim_pred, adj_pred

# ----------------------------
# 5) Training & Validation
# ----------------------------
def train_epoch(model, data_loader, optimizer, device, num_cell_types):
    model.train()
    total_loss = 0
    criterion_state = nn.KLDivLoss()
    criterion_prop = nn.SmoothL1Loss()  # for area & perimeter
    bce_loss = nn.BCEWithLogitsLoss()

    print(f"Starting training epoch with {len(data_loader)} batches...")
    for batch_idx, batch in enumerate(data_loader):
        optimizer.zero_grad()

        # Predict next state
        state_pred, area_pred, perim_pred, adj_pred = model(
            batch.x.to(device),
            batch.edge_index.to(device),
            batch.edge_attr.to(device)
        )

        # Calculate losses
        loss_state = criterion_state(state_pred, batch.next_state.to(device))
        loss_area  = criterion_prop(area_pred, batch.next_area.unsqueeze(-1).to(device))
        loss_perim = criterion_prop(perim_pred, batch.next_perim.unsqueeze(-1).to(device))
        loss_adj   = bce_loss(adj_pred, batch.next_adj.unsqueeze(-1).to(device))

        # Combine all losses + regularization
        total_batch_loss = loss_state + loss_area + loss_perim + loss_adj
        total_batch_loss += 1e-4 * sum(p.pow(2.0).sum() for p in model.parameters())  # L2 reg

        total_batch_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        optimizer.step()

        total_loss += total_batch_loss.item()

        if (batch_idx + 1) % 10 == 0:
            print(f"Batch {batch_idx+1}/{len(data_loader)} - Loss: {total_batch_loss.item():.4f}")

    avg_loss = total_loss / len(data_loader)
    print(f"Epoch complete. Average training loss: {avg_loss:.4f}")
    return avg_loss

def validate(model, data_loader, device, num_cell_types):
    model.eval()
    total_loss = 0
    print(f"Starting validation with {len(data_loader)} batches...")
    with torch.no_grad():
        for batch_idx, batch in enumerate(data_loader):
            state_pred, area_pred, perim_pred, adj_pred = model(
                batch.x.to(device),
                batch.edge_index.to(device),
                batch.edge_attr.to(device)
            )

            # Validation metrics
            loss_state = F.kl_div(state_pred, batch.next_state.to(device), reduction='batchmean')
            loss_area  = F.l1_loss(area_pred, batch.next_area.unsqueeze(-1).to(device))
            loss_perim = F.l1_loss(perim_pred, batch.next_perim.unsqueeze(-1).to(device))
            loss_adj   = F.binary_cross_entropy_with_logits(
                adj_pred, batch.next_adj.unsqueeze(-1).to(device)
            )

            total_loss += (loss_state + loss_area + loss_perim + loss_adj).item()

            if (batch_idx + 1) % 10 == 0:
                print(f"Validation Batch {batch_idx+1}/{len(data_loader)} - Cumulative Loss: {total_loss:.4f}")

    avg_loss = total_loss / len(data_loader)
    print(f"Validation complete. Average validation loss: {avg_loss:.4f}")
    return avg_loss

# ----------------------------
def k_fold_training(data_dirs, param_files, timepoints, timepoint_interval,
                    node_dim, edge_dim, hidden_dim, num_cell_types,
                    epochs=100, lr=1e-3, k_folds=5):

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    best_model = None
    best_loss = float('inf')

    # Preprocess all data
    print("Preprocessing data...")
    all_data = []
    for dir_idx, (data_dir, param_file) in enumerate(zip(data_dirs, param_files)):
        print(f"Processing directory {data_dir} with parameter file {param_file}...")
        params = parse_parameters(param_file)

        # We'll create PyG data objects for each time step
        for t in range(0, timepoints - timepoint_interval, timepoint_interval):
            print(f"Loading timepoint {t}...")
            current_graph_mat, current_prop, current_state = load_matrices(data_dir, t)
            next_graph_mat, next_prop, next_state = load_matrices(data_dir, t + timepoint_interval)

            print(f"Initializing GCA for timepoint {t}...")
            g_current = initialize_gca(current_graph_mat, current_prop, current_state, params)
            g_next    = initialize_gca(next_graph_mat,    next_prop,    next_state,    params)

            # Convert to PyG data format
            data = from_networkx(g_current)

            # (A) Create node features
            node_features = []
            for i in range(g_current.number_of_nodes()):
                # Example: combine full state distribution + area + perimeter
                state = g_current.nodes[i]["state"]  # shape [num_cell_types]
                area  = g_current.nodes[i]["area"]   # scalar
                perim = g_current.nodes[i]["perimeter"]
                node_feat = np.concatenate([state, [area, perim]])  # shape [num_cell_types+2]
                node_features.append(node_feat)

            data.x = torch.tensor(node_features, dtype=torch.float)

            # (B) Create edge features
            edge_features = []
            for (u, v) in g_current.edges():
                # Undirected => same feature for (u->v) and (v->u)
                adh   = g_current.edges[u, v]["adhesion"]
                rep_r = g_current.edges[u, v]["repulsion_radius"]
                rep_c = g_current.edges[u, v]["repulsion_coefficient"]
                
                # Features for one edge
                edge_feat = [adh, rep_r, rep_c]
                
                # Append twice
                edge_features.append(edge_feat)  # (u->v)
                edge_features.append(edge_feat)  # (v->u)

            data.edge_attr = torch.tensor(edge_features, dtype=torch.float)

            # (C) Next-step labels (for adjacency)
            # We also need to double them to match the 2*E edges in edge_index.
            adj_labels = []
            next_adj_matrix = nx.to_numpy_array(g_next)

            for (u, v) in g_current.edges():
                # 1 if still connected, else 0
                val = next_adj_matrix[u, v]
                # Append twice, once for (u->v) and again for (v->u)
                adj_labels.append(val)
                adj_labels.append(val)

            data.next_adj = torch.tensor(adj_labels, dtype=torch.float)


            # ------------- DEBUG PRINT -------------
            print("\n[DEBUG] Checking data fields for NumPy arrays or invalid types:")
            for key, val in data:
                # key is the attribute name, val is the attribute data
                if isinstance(val, np.ndarray):
                    print(f"  DEBUG: '{key}' is a NumPy array! shape={val.shape}")
                else:
                    # Print out type and shape if it's a torch tensor
                    shape_info = val.size() if hasattr(val, 'size') else None
                    print(f"  '{key}' -> {type(val)} shape={shape_info}")

            # Add to list
            all_data.append(data)

    # K-fold training
    kf = KFold(n_splits=k_folds, shuffle=True)
    all_data = np.array(all_data, dtype=object)  # to allow indexing
    for fold, (train_idx, val_idx) in enumerate(kf.split(all_data)):
        print(f"\n--- Starting Fold {fold+1}/{k_folds} ---")
        train_data_list = [all_data[i] for i in train_idx]
        val_data_list   = [all_data[i] for i in val_idx]

        train_loader = DataLoader(train_data_list, batch_size=4, shuffle=True)
        val_loader   = DataLoader(val_data_list,   batch_size=4, shuffle=False)

        print(f"Training data size: {len(train_data_list)}")
        print(f"Validation data size: {len(val_data_list)}")

        # Initialize your model with known node_dim, edge_dim
        model = GraphPredictor(node_dim, edge_dim, hidden_dim, num_cell_types).to(device)
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)

        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}/{epochs}")
            train_loss = train_epoch(model, train_loader, optimizer, device, num_cell_types)
            val_loss   = validate(model, val_loader, device, num_cell_types)
            scheduler.step(val_loss)

            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

            # Early stopping + model checkpoint
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), f"best_fold{fold}.pth")
                print(f"New best model saved for fold {fold+1} with validation loss: {best_val_loss:.4f}")
            else:
                patience_counter += 1
                if patience_counter >= 10:
                    print("Early stopping triggered.")
                    break

        # Update best overall model
        if best_val_loss < best_loss:
            best_loss = best_val_loss
            best_model = model

    print(f"\nTraining complete. Best validation loss across folds: {best_loss:.4f}")
    if best_model is not None:
        torch.save(best_model.state_dict(), "best_model.pth")
    return best_model


In [84]:
def main():
    # Example hyperparameters
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/{i}_matrix_output"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]
    timepoints     = 2000
    timepoint_intv = 400

    # Suppose each node's feature = (num_cell_types distribution) + area + perimeter => node_dim= (num_cell_types + 2)
    # Suppose each edge's feature = (adhesion, repulsion_radius, repulsion_coefficient) => edge_dim=3
    num_cell_types = 3
    node_dim       = num_cell_types + 2  # e.g. 3 + 2 = 5
    edge_dim       = 3
    hidden_dim     = 16

    best_model = k_fold_training(
        data_dirs=data_dirs,
        param_files=param_files,
        timepoints=timepoints,
        timepoint_interval=timepoint_intv,
        node_dim=node_dim,
        edge_dim=edge_dim,
        hidden_dim=hidden_dim,
        num_cell_types=num_cell_types,
        epochs=20,      # for example
        lr=1e-3,
        k_folds=2       # small for demonstration
    )

    print("Done! The best model is saved in 'best_model.pth'")

if __name__ == "__main__":
    main()

Using device: cpu
Preprocessing data...
Processing directory /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output with parameter file /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py...
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Successfully parsed parameters: {'domain_size': [60, 14], 'init_noise': 0.005, 'rng_seed': 1, 'dt': 0.25, 'tMax': 501, 'stripe_thickness': 4, 'stripe_density': 0.5, 'v0': [0.1, 1.3], 'W': array([[0.  , 0.08],
       [0.08, 0.  ]]), 'A0': [0.9, 0.9], 'P0': [3.812, 3.812], 'Dr': 50, 'kappa_A': 0.4, 'kappa_P': 0.07, 'a': 0.25, 'k': 2.5}
Loading timepoint 0...
Loading matrices for timepoint 0 from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output
Loading matrices for timepoint 400 from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_m

TypeError: DataLoader found invalid type: '<class 'numpy.ndarray'>'

In [ ]:
if __name__ == "__main__":
    # Directories and parameter files
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/{i}_matrix_output"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    # Run the main training process
    trained_model = main(data_dirs, param_files)

    # Save the final trained model
    torch.save(trained_model.state_dict(), "final_trained_model.pth")
    print("Final model saved to 'final_trained_model.pth'")

In [ ]:
import os

# Check if the files exist
directory = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output"
timepoint = 0  # Replace with the correct timepoint
required_files = [
    f"{timepoint}_graph_mat.npy",
    f"{timepoint}_properties_mat.npy",
    f"{timepoint}_state_mat.npy"
]

for file in required_files:
    file_path = os.path.join(directory, file)
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
    else:
        print(f"File exists: {file_path}")